In [ ]:
import pandas as pd
from docx import Document
import json
import os
from openpyxl import load_workbook
import openpyxl
import time
from openai import OpenAI
import ast
import openpyxl
import re
import itertools
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks

In [ ]:
def read_docx(number):
    file_path = f'../data/竺可楨全集第0{number}卷.docx'
    doc = Document(file_path)
    full_text = []
    for para in doc.paragraphs:
        para_text = para.text.strip()
        if para_text == "":
            continue  # 跳过这个段落
        if not para_text.endswith("。"):
            para_text += "。"
        full_text.append(para_text)
    return '\n'.join(full_text)
    
def ner(sentence):
    sentence_list=[]
    result = ner_pipeline(sentence)
    result=result.get('output')
    for res in result:
        if res.get('type')=='GPE':
            sentence_list.append(res.get('span'))
    return sentence_list

In [ ]:
def process_entry(entry):
    result = openai(entry)
    try:
        result_data = json.loads(result)
        loc_list = list(set(result_data.get('location', [])))
        return loc_list
    except Exception as e:
        print(e, result)
        return None

def write_to_excel(file_path, data, headers):
    workbook = openpyxl.Workbook()
    sheet = workbook.active
    sheet.append(headers)
    for row in data:
        sheet.append(row)
    workbook.save(file_path)


In [ ]:
import re
import os

date_pattern = r'(\d*\s*\d*\s*月\s*\d*\s*\d*\s*日\s*星\s*期\s*[一二三四五六日])'
entry_pattern = r'(\d*\s*\d*\s*月\s*\d*\s*\d*\s*日\s*星\s*期\s*[一二三四五六日].*?)(?=\d*\s*\d*\s*月\s*\d*\s*\d*\s*日\s*星\s*期\s*[一二三四五六日]|$)'

def load_keywords(keyword_file):

    with open(keyword_file, 'r', encoding='utf-8') as file:
        keywords = [line.strip() for line in file if line.strip()]
    return keywords

def search_keywords_in_diary(line,keywords):

    matched_keywords = []
    
    for keyword in keywords:
        if re.search(re.escape(keyword), line, re.IGNORECASE):
            matched_keywords.append(keyword)
    return matched_keywords
    
def get_diary(number,keywords):
    all_excel_rows = []
    text = read_docx(number)
    entries = re.findall(entry_pattern, text, re.DOTALL)
    
    for entry in entries:
        date_match = re.match(date_pattern, entry)
        if date_match:
            full_date = date_match.group(0).strip()
            sentences = entry.split('\n')
            sentences = list(filter(None, sentences))
            header=sentences[0]
            header = re.sub(r'[〔〕<>〈〉\s]', '', header)
            loc_list=ner(header)
            
            for sentence in sentences:
                lines = sentence.split('。')
                for line in lines:
                    if not line or re.search(r'^(接|寄)', line):
                        continue
                    matched_keywords=search_keywords_in_diary(line,keywords)
                    if matched_keywords:
                        all_excel_rows.extend([[full_date,header," ".join(loc_list),line,sentence," ".join(matched_keywords)]])

    write_to_excel(f'Re-{number}.xlsx', all_excel_rows, ["date", "header","loc_list","line","sentence","matched_keywords"])
    
    print("success")



def main():
    # 关键词文件路径
    keyword_file = 'qx_keyword.txt'
    # 加载关键词
    keywords = load_keywords(keyword_file)
    
    print(f"Loaded keywords: {keywords}")
                        
    get_diary(10, keywords)

if __name__ == "__main__":
    main()